In [ ]:
import os
import sys
sys.path.append("../stable-baselines3")
sys.path.append("..")
from torch.nn.modules.activation import F
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gymnasium as gym
from logger import setup_logger
from livestockEnvV2 import load_datas,LivestockEnvConfig
from stable_baselines3 import PPO_action_mask_v2
from stable_baselines3.common.env_util import make_vec_env
from gymnasium.envs.registration import register
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecCheckNan
from AttentionPolicy import CustomAttentionPolicy

register(
    id='LivestockEnv-v2',
    entry_point='livestockEnvV2:LivestockEnv',
)
country = 'cn'
config = LivestockEnvConfig(country, 
                            Reward_priority=[4, 4, 2, 1], 
                            thresholds=[31, 0], 
                            mobility_ratio=0.25,
                            max_steps=8000)

# 创建并包装环境
env = make_vec_env('LivestockEnv-v2', n_envs=1, env_kwargs={'config': config})
env = VecCheckNan(env, raise_exception=True)
eval_callback = EvalCallback(env, best_model_save_path=f'../logs/v3/{country}/',
                             log_path='./logs/', eval_freq=config.max_steps,
                             deterministic=False, render=False)
from typing import Callable

def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func
model = PPO_action_mask_v2(CustomAttentionPolicy, 
                        env, 
                        batch_size=4, 
                        verbose=1, 
                        tensorboard_log='./board/',
                        seed=42,
                        kwargs={'country':country},
                        learning_rate=linear_schedule(2e-2),
                        n_steps=2**13,
                        )

In [ ]:
model.learn(total_timesteps=100000,tb_log_name = f"{country}PPO_v3",callback=eval_callback)

In [3]:
# model.save(f"./checkpoints/{country}_ppo_livestockV1.0")